# Machine Learning-Based Buyer Segmentation and Investment Profiling
## Real Estate Market Intelligence — Full Pipeline Walkthrough

**Client:** Parcl Co. Limited | **Program:** Unified Mentor

This notebook walks through the entire project end to end: data audit, cleaning,
feature engineering, EDA, clustering, validation, and robustness testing.
Every cell below is **actually executed against the real project data and code**
in `src/` — these are the same numbers and charts the Streamlit dashboard and
research paper use, not a separately-maintained illustrative copy.

**Before running:** make sure you've run the pipeline scripts at least once
(`python src/01_Data_Preprocessing.py`, `python src/03_Buyer_Segmentation_Clustering.py`, `python src/02_Exploratory_Data_Analysis.py`,
`python src/04_Model_Robustness_Validation.py`) so the files this notebook reads actually exist.


## 0. Setup — locate the project root

This cell works whether you open this notebook from inside `notebooks/` (the default in Jupyter/VS Code) or from the project root itself (e.g. some Colab setups) — it checks both and fails with a clear message rather than a silent wrong path.


In [1]:
# ============================================================
# 0. SETUP — robust project discovery + source-module loading
# ============================================================
import sys
import importlib.util
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

def find_project_root(start=None):
    """Locate the real project root from a VS Code/Jupyter notebook."""
    start = Path(start or Path.cwd()).resolve()
    def is_project_root(p):
        return ((p / "data" / "raw" / "Client_Master_Raw_Data.csv").is_file()
                and (p / "data" / "raw" / "Property_Transactions_Raw_Data.csv").is_file()
                and (p / "src").is_dir())
    for candidate in [start, *start.parents]:
        if is_project_root(candidate): return candidate
    for clients_file in start.rglob("Client_Master_Raw_Data.csv"):
        candidate = clients_file.parent.parent.parent
        if is_project_root(candidate): return candidate.resolve()
    raise FileNotFoundError(
        "Project root not found. Expected data/raw/Client_Master_Raw_Data.csv, "
        "data/raw/Property_Transactions_Raw_Data.csv and a src folder.\n"
        f"Current working directory: {start}"
    )

BASE = find_project_root()
SRC = BASE / "src"
RAW = BASE / "data" / "raw"
OUTPUTS = BASE / "outputs"
if str(SRC) not in sys.path: sys.path.insert(0, str(SRC))

def load_src_module(alias, candidates, keywords=()):
    """Load a project source module using known names, then a keyword fallback."""
    path = next((SRC / name for name in candidates if (SRC / name).is_file()), None)
    if path is None and keywords:
        matches = sorted(p for p in SRC.glob("*.py") if all(k.lower() in p.name.lower() for k in keywords))
        path = matches[0] if matches else None
    if path is None:
        available = ", ".join(p.name for p in sorted(SRC.glob("*.py")))
        raise FileNotFoundError(f"Could not find source module for {alias} in {SRC}. Available files: {available}")
    spec = importlib.util.spec_from_file_location(alias, path)
    if spec is None or spec.loader is None: raise ImportError(f"Could not load source module: {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[alias] = module
    spec.loader.exec_module(module)
    return module

preprocessing = load_src_module("preprocessing", ["01_Data_Preprocessing.py", "preprocessing.py"], ("preprocessing",))
eda = load_src_module("eda", ["02_Exploratory_Data_Analysis.py", "eda.py"], ("exploratory",))
clustering = load_src_module("clustering", ["03_Buyer_Segmentation_Clustering.py", "03_Buyer_Segmentation_Clustering.py", "clustering.py"], ("clustering",))

def show_project_image(filename, figsize=(10, 5), title=None):
    """Display a generated project figure without crashing if it is absent."""
    import matplotlib.image as mpimg
    path = OUTPUTS / "figures" / filename
    if not path.is_file():
        print(f"Figure not found: {path}")
        return
    plt.figure(figsize=figsize)
    plt.imshow(mpimg.imread(path)); plt.axis("off")
    if title: plt.title(title)
    plt.show()

print("Project root:", BASE)
print("Python:", sys.version.split()[0])
print("Preprocessing:", preprocessing.__file__)
print("EDA:", eda.__file__)
print("Clustering:", clustering.__file__)
print("Setup complete.")


Project root: C:\Users\Dell\Downloads\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence
Python: 3.11.16
Preprocessing: C:\Users\Dell\Downloads\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence\src\01_Data_Preprocessing.py
EDA: C:\Users\Dell\Downloads\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence\src\02_Exploratory_Data_Analysis.py
Clustering: C:\Users\Dell\Downloads\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence\Real-Estate-Buyer-Segmentation-and-Investment-Intelligence\src\03_Buyer_Segmentation_Clustering.py
Setup complete.


## 1. Data Audit — what are we actually working with?

Before touching the data, confirm its real shape, types, and relationships rather than assuming the schema described in the project brief.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

clients_raw = pd.read_csv(BASE / "data" / "raw" / "Client_Master_Raw_Data.csv")
properties_raw = pd.read_csv(BASE / "data" / "raw" / "Property_Transactions_Raw_Data.csv")

print("Client_Master_Raw_Data.csv:", clients_raw.shape)
print("Property_Transactions_Raw_Data.csv:", properties_raw.shape)
print()
print("Client_Master_Raw_Data.csv columns:", list(clients_raw.columns))
print("Property_Transactions_Raw_Data.csv columns:", list(properties_raw.columns))


Client_Master_Raw_Data.csv: (2000, 12)
Property_Transactions_Raw_Data.csv: (10000, 9)

Client_Master_Raw_Data.csv columns: ['client_id', 'client_type', 'first_name', 'last_name', 'date_of_birth', 'gender', 'country', 'region', 'acquisition_purpose', 'satisfaction_score', 'loan_applied', 'referral_channel']
Property_Transactions_Raw_Data.csv columns: ['listing_id', 'tower_number', 'transaction_date', 'unit_category', 'unit_number', 'floor_area_sqft', 'sale_price', 'listing_status', 'client_ref']


In [3]:
print("Missing values in Client_Master_Raw_Data.csv:", clients_raw.isnull().sum().sum())
print("Missing values in Property_Transactions_Raw_Data.csv:", properties_raw.isnull().sum().sum())
print("Duplicate client_id:", clients_raw['client_id'].duplicated().sum())
print("Duplicate listing_id:", properties_raw['listing_id'].duplicated().sum())
print()
print("acquisition_purpose values:", clients_raw['acquisition_purpose'].unique())
print("(Note: the original project brief lists 'Personal use' - the actual value is 'Home')")
print()
print("date_of_birth sample (note the MIXED formats):")
print(clients_raw['date_of_birth'].head(6).tolist())


Missing values in Client_Master_Raw_Data.csv: 0
Missing values in Property_Transactions_Raw_Data.csv: 2695
Duplicate client_id: 0
Duplicate listing_id: 0

acquisition_purpose values: <ArrowStringArray>
['Home', 'Investment']
Length: 2, dtype: str
(Note: the original project brief lists 'Personal use' - the actual value is 'Home')

date_of_birth sample (note the MIXED formats):
['05-11-1968', '11/26/1962', '04-07-1959', '11/25/1959', '2/28/1976', '03-06-1957']


In [4]:
sold = properties_raw[properties_raw['listing_status'] == 'Sold']
available = properties_raw[properties_raw['listing_status'] == 'Available']
print(f"Sold listings: {len(sold)} (all have a client_ref: {sold['client_ref'].notnull().all()})")
print(f"Available listings: {len(available)} (all have NO client_ref: {available['client_ref'].isnull().all()})")
print()
purchases_per_client = sold['client_ref'].value_counts()
print(f"Every one of {clients_raw.shape[0]} clients has at least one purchase: "
      f"{purchases_per_client.min() >= 1 and len(purchases_per_client) == clients_raw.shape[0]}")
print(f"Min purchases per client: {purchases_per_client.min()}, Max: {purchases_per_client.max()}")
print()
print("=> A naive clients.merge(properties, on='client_id') would duplicate each client's")
print("   demographic row once per property purchased. Property_Transactions_Raw_Data.csv must be aggregated")
print("   to one row per client FIRST (see src/01_Data_Preprocessing.py) before joining.")


Sold listings: 7305 (all have a client_ref: True)
Available listings: 2695 (all have NO client_ref: True)

Every one of 2000 clients has at least one purchase: True
Min purchases per client: 3, Max: 13

=> A naive clients.merge(properties, on='client_id') would duplicate each client's
   demographic row once per property purchased. Property_Transactions_Raw_Data.csv must be aggregated
   to one row per client FIRST (see src/01_Data_Preprocessing.py) before joining.


## 2. Data Cleaning & Feature Engineering

Reusing the exact functions from `src/01_Data_Preprocessing.py` - not a separate copy - so this notebook can never silently drift from what the dashboard and paper actually use.


In [5]:
master, clients_clean, properties_clean = preprocessing.build_master_table()
print("Master client-level table:", master.shape)
print(master[["client_id","client_type","age","total_purchases","total_spend",
              "avg_purchase_price","loan_flag","investment_flag","is_company"]].head(5).to_string())


Master client-level table: (2000, 29)
  client_id client_type   age  total_purchases  total_spend  avg_purchase_price  loan_flag  investment_flag  is_company
0     C0001  Individual  57.6                4   1246764.72       311691.180000          1                0           0
1     C0002  Individual  63.1                5   1841095.93       368219.186000          0                0           0
2     C0003  Individual  66.7                5   1661457.59       332291.518000          1                0           0
3     C0004  Individual  66.1                6   1608263.51       268043.918333          0                0           0
4     C0005     Company  49.8               13   3653385.38       281029.644615          0                1           1


In [6]:
print("Engineered columns added beyond the raw client fields:")
new_cols = [c for c in master.columns if c not in clients_raw.columns]
for c in new_cols:
    print(" -", c)


Engineered columns added beyond the raw client fields:
 - age
 - loan_flag
 - investment_flag
 - is_company
 - total_purchases
 - total_spend
 - avg_purchase_price
 - max_purchase_price
 - avg_floor_area
 - total_floor_area
 - n_towers
 - first_purchase_date
 - last_purchase_date
 - pct_apartment
 - pct_office
 - is_repeat_buyer
 - purchase_span_days


## 3. Exploratory Data Analysis

Each chart answers a specific business question, reusing `src/02_Exploratory_Data_Analysis.py`'s exact functions.


In [7]:
eda.q_who_are_the_buyers(master)
show_project_image("01_Buyer_Mix_by_Client_Type.png", figsize=(6,5), title="Who are the buyers?")


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\4033395867.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
eda.q_country_distribution(master)
show_project_image("09_Buyer_Distribution_by_Country.png", figsize=(8,5), title="Buyer distribution by country")
print("USA share of clients:", round((master["country"] == "USA").mean() * 100, 1), "%")
print("Small-country rates should be treated as indicative because their sample sizes are much smaller.")


USA share of clients: 76.9 %
Small-country rates should be treated as indicative because their sample sizes are much smaller.


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\4033395867.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
eda.q_investment_by_client_type(master)
show_project_image("02_Acquisition_Purpose_by_Client_Type.png", figsize=(7,5), title="Acquisition purpose by client type")


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\4033395867.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
eda.q_loan_by_clienttype(master)
show_project_image("04_Financing_Behavior_by_Client_Type.png", figsize=(6,5), title="Financing behavior by client type")


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\4033395867.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Clustering — K-Means, Elbow Method, Silhouette Score, Hierarchical validation

Reusing `src/03_Buyer_Segmentation_Clustering.py` directly.


In [11]:
df = clustering.load_master()
X, preprocessor, feature_names = clustering.build_feature_matrix(df)
print("Clustering feature space:", feature_names)
print("Feature matrix shape:", X.shape)


Clustering feature space: ['age', 'satisfaction_score', 'total_purchases', 'total_spend', 'avg_purchase_price', 'avg_floor_area', 'n_towers', 'pct_apartment', 'purchase_span_days', 'loan_flag', 'investment_flag', 'is_company']
Feature matrix shape: (2000, 12)


In [12]:
ks, inertias, sil_scores = clustering.evaluate_k_range(X, 2, 10)
for k, s in zip(ks, sil_scores):
    marker = "  <-- selected (see reasoning below)" if k == 4 else (
        "  <-- max silhouette" if s == max(sil_scores) else ""
    )
    print(f"k={k}: silhouette={s:.4f}{marker}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[1].plot(ks, sil_scores, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()


k=2: silhouette=0.1381
k=3: silhouette=0.1451  <-- max silhouette
k=4: silhouette=0.1405  <-- selected (see reasoning below)
k=5: silhouette=0.1286
k=6: silhouette=0.1308
k=7: silhouette=0.1314
k=8: silhouette=0.1322
k=9: silhouette=0.1331
k=10: silhouette=0.1352


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\2793527127.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
print("Silhouette peaks at k=3 (0.1451) and is nearly flat from k=3 to k=8 - there is no")
print("sharply separated natural cluster structure in this data. k=4 is selected anyway")
print("because it isolates corporate buyers (is_company ~1.0 in one cluster, ~0 in the")
print("others) into their own operationally meaningful segment, at only a small")
print("silhouette cost versus k=3. This is a business-interpretability judgment call,")
print("not a purely statistical one - and it is disclosed as such rather than hidden.")


Silhouette peaks at k=3 (0.1451) and is nearly flat from k=3 to k=8 - there is no
sharply separated natural cluster structure in this data. k=4 is selected anyway
because it isolates corporate buyers (is_company ~1.0 in one cluster, ~0 in the
others) into their own operationally meaningful segment, at only a small
silhouette cost versus k=3. This is a business-interpretability judgment call,
not a purely statistical one - and it is disclosed as such rather than hidden.


In [14]:
from sklearn.metrics import silhouette_score

final_model, labels = clustering.fit_final_kmeans(X, 4)
df["cluster"] = labels
final_silhouette = silhouette_score(X, labels)
ari = clustering.compare_with_hierarchical(X, 4, labels)

print(f"Final model: k=4, silhouette={final_silhouette:.4f}, ARI vs Hierarchical={ari:.4f}")
print()
print("Cluster sizes:")
print(df["cluster"].value_counts().sort_index())


Final model: k=4, silhouette=0.1405, ARI vs Hierarchical=0.2669

Cluster sizes:
cluster
0    446
1    640
2    812
3    102
Name: count, dtype: int64


In [15]:
show_project_image("13_Hierarchical_Clustering_Cross_Check.png", figsize=(10,4), title="Hierarchical clustering dendrogram (validation)")


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\4033395867.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Cluster Profiling & Business Segment Naming

Segment names are derived by ranking clusters against EACH OTHER on the metrics that separate them - not from fixed thresholds or the project brief's assumed labels.


In [16]:
profile_cols = ["age","satisfaction_score","total_purchases","total_spend",
                 "avg_purchase_price","avg_floor_area","pct_apartment",
                 "loan_flag","investment_flag","is_company"]
profile = df.groupby("cluster")[profile_cols].mean().round(3)
profile["size"] = df["cluster"].value_counts().sort_index()
segment_names = clustering.name_segments(profile)
profile["segment_name"] = profile.index.map(segment_names)
print(profile.to_string())


            age  satisfaction_score  total_purchases  total_spend  avg_purchase_price  avg_floor_area  pct_apartment  loan_flag  investment_flag  is_company  size                   segment_name
cluster                                                                                                                                                                                          
0        54.652               3.007            3.123  1353463.441          432123.964        1415.712          0.834      0.343            0.296       0.000   446    Premium / High-Value Buyers
1        57.713               3.156            4.358  1549452.754          358204.326        1181.958          0.859      0.356            0.278       0.002   640  High-Volume / Frequent Buyers
2        55.460               2.938            3.393   983920.294          291826.136         973.274          0.855      0.384            0.333       0.000   812    Value-Conscious Home Buyers
3        47.582               

In [17]:
show_project_image("11_Final_Buyer_Segment_Distribution.png", figsize=(11,4), title="Final buyer segment distribution")


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\4033395867.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
show_project_image("12_Final_Segment_Profile.png", figsize=(14,4), title="Final segment profile")


C:\Users\Dell\AppData\Local\Temp\ipykernel_20312\4033395867.py:68: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Robustness & Stability Analysis

Beyond a single Elbow/Silhouette pass: alternative feature strategies, K-Means
stability across random seeds, and stability under 80%-resample bootstrapping.
Reusing `src/04_Model_Robustness_Validation.py` directly.


In [19]:
import json
summary = json.loads((BASE / "outputs" / "tables" / "11_Model_Robustness_Summary.json").read_text())
for k, v in summary.items():
    print(f"{k}: {v}")


production_strategy: Current production
production_k: 4
production_silhouette: 0.14049556921628967
best_silhouette_over_tested_strategies: 0.17175727342312058
best_strategy_by_silhouette: Balanced behavior
best_k_by_silhouette: 3
seed_stability_mean_ari: 0.7309932931703061
seed_stability_min_ari: 0.5155625363188454
bootstrap_mean_ari: 0.5826182235309443
bootstrap_min_ari: 0.26328634707530135


In [20]:
seed_stab = pd.read_csv(BASE / "outputs" / "tables" / "09_KMeans_Seed_Stability.csv")
print(seed_stab.to_string(index=False))
print()
print("Note the pattern: seed 0 and seed 123 agree perfectly (ARI=1.0) with each other,")
print("seeds 7/21/42/84 agree closely with each other (ARI 0.94-0.99), but the two")
print("GROUPS only agree with each other at ARI~0.52. This is bimodal convergence -")
print("two distinct stable solutions - not uniform noise around one solution. The")
print("production model uses random_state=42, which falls in the 4-seed majority group.")


 seed_a  seed_b      ari
      0       7 0.523047
      0      21 0.516814
      0      42 0.526850
      0      84 0.515563
      0     123 1.000000
      7      21 0.991646
      7      42 0.991870
      7      84 0.944338
      7     123 0.523047
     21      42 0.983572
     21      84 0.949184
     21     123 0.516814
     42      84 0.939742
     42     123 0.526850
     84     123 0.515563

Note the pattern: seed 0 and seed 123 agree perfectly (ARI=1.0) with each other,
seeds 7/21/42/84 agree closely with each other (ARI 0.94-0.99), but the two
GROUPS only agree with each other at ARI~0.52. This is bimodal convergence -
two distinct stable solutions - not uniform noise around one solution. The
production model uses random_state=42, which falls in the 4-seed majority group.


In [21]:
corr = pd.read_csv(BASE / "outputs" / "tables" / "05_Clustering_Feature_Correlation.csv", index_col=0)
print("Selected feature correlations (the ones that matter for interpreting the segments):")
print(corr.loc[['total_purchases','avg_purchase_price'], ['n_towers','avg_floor_area','total_spend']].round(2).to_string())
print()
print("total_purchases<->n_towers = 0.93 and avg_purchase_price<->avg_floor_area = 0.96:")
print("five of the nine numeric clustering features substantially encode the same")
print("underlying 'transaction scale' dimension, which is why the segments separate")
print("mainly by spend/purchase scale rather than by financing or investment behavior")
print("(loan_flag, investment_flag, is_company all correlate <=0.08 with everything).")


Selected feature correlations (the ones that matter for interpreting the segments):
                    n_towers  avg_floor_area  total_spend
total_purchases         0.93           -0.13         0.71
avg_purchase_price     -0.12            0.96         0.60

total_purchases<->n_towers = 0.93 and avg_purchase_price<->avg_floor_area = 0.96:
five of the nine numeric clustering features substantially encode the same
underlying 'transaction scale' dimension, which is why the segments separate
mainly by spend/purchase scale rather than by financing or investment behavior
(loan_flag, investment_flag, is_company all correlate <=0.08 with everything).


## 7. Brief-Compliant Encoding — Tested Directly, Not Skipped

The original project brief's Step 2 specifies One-Hot/Label Encoding of
client_type, region, acquisition_purpose, referral_channel, and country. That
literal approach was implemented and measured (`src/05_Brief_Encoding_Validation.py`),
not assumed to be worse.


In [22]:
import subprocess

result = subprocess.run(
    [sys.executable, str(BASE / "src" / "05_Brief_Encoding_Validation.py")],
    capture_output=True,
    text=True,
    cwd=str(BASE),
)

print(result.stdout[-4000:] if result.stdout else "(No standard output.)")
if result.returncode != 0:
    print("\n--- STDERR ---")
    print(result.stderr[-4000:] if result.stderr else "(No error output.)")
    raise RuntimeError(
        f"05_Brief_Encoding_Validation.py exited with code {result.returncode}"
    )


BRIEF-COMPLIANT ENCODING TEST
Literal implementation of the brief's Step 2: One-Hot Encoding of
client_type, region, acquisition_purpose, referral_channel, country

One-Hot feature space: 76 columns (region alone contributes 57 of them)
 k  silhouette     inertia
 2    0.163385 7230.209869
 3    0.149966 6285.973899
 4    0.133264 5747.074602
 5    0.133453 5450.347817
 6    0.130943 5189.191983
 7    0.128882 4998.996440
 8    0.126584 4820.980831
 9    0.129198 4663.839760
10    0.127426 4530.935024

Label-Encoded feature space: 7 columns (['age', 'satisfaction_score', 'client_type_le', 'region_le', 'acquisition_purpose_le', 'referral_channel_le', 'country_le'])
 k  silhouette      inertia
 2    0.412646 11975.209288
 3    0.215548 10082.926364
 4    0.201662  8873.213999
 5    0.218032  7620.303382
 6    0.207326  6976.923871
 7    0.228048  6512.281467
 8    0.211502  6077.203475
 9    0.216330  5779.094863
10    0.222515  5490.323009

SUMMARY — best silhouette per approach (raw nu

## 8. Final Segments & Business Recommendations

| Segment | Size | Defining trait | Recommended action |
|---|---|---|---|
| Value-Conscious Home Buyers | 812 (41%) | Lowest spend/price among individuals | Affordability-focused messaging, starter financing |
| High-Volume / Frequent Buyers | 640 (32%) | Highest purchase frequency | Loyalty/portfolio incentives, early access |
| Premium / High-Value Buyers | 446 (22%) | Highest average purchase price | Exclusivity, white-glove service over discounting |
| Corporate Buyers | 102 (5%) | Company clients | B2B sales motion, bulk/multi-unit packages |

## 9. Honest Limitations

- Silhouette scores are modest (0.12-0.15): buyer behavior varies continuously, not in sharp groups.
- Seed stability is bimodal (see Section 6) - a different arbitrary seed could plausibly ship a qualitatively different clustering.
- The client base is 76.9% USA; country-level comparisons for small countries (e.g. Denmark, n=15) are statistically noisy.
- Five of nine clustering features are highly intercorrelated (transaction-scale dimension), which shapes what separates the segments.
- The project brief's own literal encoding approach (Section 7) scores a numerically higher silhouette but produces degenerate clusters (one reproduces an existing column, the other splits on a single field) - silhouette alone cannot be trusted without inspecting cluster composition.
- No marketing-response/campaign-cost data exists to validate the business recommendations' ROI directly.

## Conclusion

This project replaced an assumed, template-driven buyer segmentation with one derived directly from Parcl's actual client and transaction data, validated with Elbow/Silhouette/Hierarchical methods plus seed and bootstrap robustness testing, and delivered through an interactive, filterable Streamlit dashboard. The clustering's modest statistical separation is disclosed rather than hidden — an honest finding about the data, not a shortcoming in the methodology.
